[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/MFM/blob/main/Quantlets/Ch_19/MFM_ch19_integrated_bitcoin/MFM_ch19_integrated_bitcoin.ipynb)

# MFM_ch19_integrated_bitcoin

Integrated case study on Bitcoin (daily closes 2014-2026 and 5-minute bars on UTC days 2024-2026): Hill tail index and volatility clustering (Chapter 1), realised variance from 5-minute bars and expanding-window log-HAR forecasts (Chapter 9), a one-day VaR 1% from the forecast against historical simulation with Kupiec and Christoffersen backtests (Chapter 8), spread estimators and Amihud illiquidity (Chapter 10), and the PSY GSADF bubble test with BSADF date-stamping (Chapter 17).

*Modelling Financial Markets (MFM), Daniel Traian Pele, Bucharest University of Economic Studies*

> **Working in Google Colab? Save your own copy first.**
> The Colab link opens a temporary copy: when you close the tab or the session times out, your changes are lost.
> Use **File → Save a copy in Drive** (it goes to the *Colab Notebooks* folder of your ASE Google Drive) and work in that copy.
> For the team project, **File → Save a copy in GitHub** puts the notebook directly in your team's repository.
> Files you create while the notebook runs (CSV, charts) are also temporary: set `SAVE_TO_DRIVE = True` in the next cell to keep them.

In [ ]:
# Optional (Colab only): keep the files you create in Google Drive
SAVE_TO_DRIVE = False   # set to True, run this cell and allow access to your Drive

import os
OUT_DIR = '.'
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT_DIR = '/content/drive/MyDrive/MFM/Chapter_19'
    os.makedirs(OUT_DIR, exist_ok=True)
print('Save your outputs to:', os.path.abspath(OUT_DIR))


In [ ]:
# Colab: !pip install arch
import os
import numpy as np
import pandas as pd
try:
    import arch  # noqa: F401
except ModuleNotFoundError as exc:
    if exc.name != 'arch':
        raise
    import subprocess
    import sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'arch'], check=True)
from IPython.display import Image, display
import warnings
warnings.filterwarnings('ignore')


## The code of the chapters

`integrated_cases.py` imports the modules of each chapter from its own folder and never writes the charts or results of another chapter.

In [ ]:
# The case study calls the code of several chapters of the course (Quantlets/Ch_NN). Inside a clone of the
# course repository the local files are used; otherwise the modules are downloaded from GitHub.
import sys
import urllib.request
RAW = 'https://raw.githubusercontent.com/danpele/MFM/main/Quantlets/'
FILES = ["slide_fit.py", "slide_fit_sizes.json", "Ch_19/integrated_cases.py", "Ch_19/inference19.py", "Ch_01/generate_all_charts.py", "Ch_01/mfm_data.py", "Ch_07/risk_measures.py", "Ch_08/mfm_data.py", "Ch_09/mfm_data.py", "Ch_09/rv_tools.py", "Ch_09/ch9_forecasts_btc.csv", "Ch_10/generate_all_charts.py", "Ch_10/mfm_data.py", "Ch_10/micro.py", "Ch_17/bubbles.py", "Ch_17/generate_all_charts.py", "Ch_17/mfm_data.py"]
QL = os.environ.get('MFM_QUANTLETS') or os.path.abspath(os.path.join('..', '..'))
if not os.path.exists(os.path.join(QL, 'Ch_19', 'integrated_cases.py')):
    QL = os.path.abspath('mfm_quantlets')
    for f in FILES:
        path = os.path.join(QL, f)
        os.makedirs(os.path.dirname(path), exist_ok=True)
        if not os.path.exists(path):
            urllib.request.urlretrieve(RAW + f, path)
sys.path.insert(0, os.path.join(QL, 'Ch_19'))
os.environ['MFM_CHART_DIR'] = OUT_DIR     # the chart (PDF, PNG) goes to OUT_DIR
os.environ['MFM_OUT_DIR'] = OUT_DIR       # and so do the CSV series
import integrated_cases as IC


## Steps 1-3, Chapters 1, 9 and 8: tails, realised variance and the VaR 1%

Realised variance is the sum of squared 5-minute returns of each complete UTC day; the log-HAR model is re-estimated every day on all earlier days and its forecast gives VaR 1% = 2.326 times the forecast volatility, compared with historical simulation on the last 500 daily returns.

## Steps 4-5, Chapters 10 and 17: liquidity and bubbles

Spread estimators from daily data and the Amihud price impact per USD 1 million traded; the GSADF test on weekly log prices with Monte Carlo critical values, and the BSADF sequence that dates the explosive episodes (at least 7 weeks above the 95% critical value).

In [ ]:
R, D, W, M = IC.bitcoin_case()
IC.fig_bitcoin(D, W, M, R['step5']['episodes'])
display(Image(os.path.join(OUT_DIR, 'ch19_int_bitcoin.png')))

## The numbers behind the chart

In [ ]:
print({k: round(v, 3) if isinstance(v, float) else v for k, v in R['step1'].items()})
pd.DataFrame({k: R['step3'][k] for k in ('har', 'hs')}).T[['T', 'x', 'expected', 'p_uc', 'p_ind', 'p_cc']].round(4)

In [ ]:
print('Liquidity:', {k: round(v, 4) if isinstance(v, float) else v for k, v in R['step4'].items()})
print('GSADF', round(R['step5']['gsadf'], 2), '95% critical value', round(R['step5']['cv95'], 2))
pd.DataFrame(R['step5']['episodes'])

Figure file: [ch19_int_bitcoin](./ch19_int_bitcoin.png). The analysis displays its freshly generated figure inline and saves PDF and PNG files in OUT_DIR.